In [1]:
pip install email

  Using cached email-4.0.2.tar.gz (1.2 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 353, in <module>
          main()
        File "c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 335, in main
          json_out['return_val'] = hook(**hook_input['kwargs'])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 118, in get_requires_for_build_wheel
          return hook(config_settings)
                 ^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\admin\AppData\Local\Temp\pip-build-env-_v6ol8fi\

In [2]:
pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install python-docx

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install openai==0.28.1

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install PyMuPDF

In [3]:
pip install pdfplumber


Note: you may need to restart the kernel to use updated packages.


In [12]:
import email
from email import policy
from email.parser import BytesParser
from bs4 import BeautifulSoup  # HTML 본문 처리를 위한 라이브러리

def extract_email_body_recursive(part):
    """재귀적으로 파트를 탐색하여 본문 추출"""
    body = ""
    if part.is_multipart():
        for subpart in part.iter_parts():
            body += extract_email_body_recursive(subpart)  # 하위 파트를 재귀적으로 탐색
    else:
        if part.get_content_type() == 'text/plain':
            body += part.get_payload(decode=True).decode(part.get_content_charset(), errors="replace")
        elif part.get_content_type() == 'text/html':
            html_content = part.get_payload(decode=True).decode(part.get_content_charset(), errors="replace")
            soup = BeautifulSoup(html_content, 'html.parser')
            body += soup.get_text()
    return body

def extract_email_body(eml_file_path):
    """이메일 파일에서 제목과 본문 추출"""
    with open(eml_file_path, 'rb') as f:
        msg = BytesParser(policy=policy.default).parse(f)

    # 이메일 제목 추출
    subject = msg['subject']
    
    # 이메일 본문 추출
    body = ""
    if msg.is_multipart():
        body = extract_email_body_recursive(msg)
    else:
        body = msg.get_payload(decode=True).decode(msg.get_content_charset(), errors="replace")
    
    return subject, body

# .eml 파일 경로 설정
eml_file_path = r"C:\Users\admin\Documents\악성 이메일 모음\_!____.eml"  # 경로 수정
# 이메일 제목과 본문 추출
email_title, email_body = extract_email_body(eml_file_path)
print(f"Email Title: {email_title}\nEmail Body: {email_body[:500]}")  # 본문의 앞 500자만 출력


Email Title: !,정규리#
Email Body: 직원 구하는시는것 보고 연락드립니다.
성실한 지원자입니다.
나의 마음가짐이 무한한 가능성을 열리게한다고 생각합니다

일을 하면서 다른 직원분들과 일을 진행해야 하는지도 많이 배웠습니다
일을 했어서 업무적응은 문제없다고 생각합니다

면접은 오늘이라도 가능한 상황입니다
비밀번호는 230124입니다

잘부탁드립니다
직원 구하는시는것 보고 연락드립니다.성실한 지원자입니다.나의 마음가짐이 무한한 가능성을 열리게한다고 생각합니다일을 하면서 다른 직원분들과 일을 진행해야 하는지도 많이 배웠습니다일을 했어서 업무적응은 문제없다고 생각합니다면접은 오늘이라도 가능한 상황입니다비밀번호는 230124입니다잘부탁드립니다



In [13]:
import pdfplumber
import os
import re
import win32com.client as win32  # Microsoft Word COM 객체 사용
from email import policy
from email.parser import BytesParser
from bs4 import BeautifulSoup  # HTML 본문 처리를 위한 라이브러리
import io

def extract_attachments(eml_file_path):
    """첨부파일을 추출하고, 없으면 본문에 있는 링크를 출력"""
    attachment_text = ""
    attachment_found = False  # 첨부파일이 있는지 여부를 추적
    try:
        with open(eml_file_path, 'rb') as f:
            msg = BytesParser(policy=policy.default).parse(f)

        if msg.is_multipart():
            for part in msg.iter_parts():
                # 첨부파일이 있는지 확인
                if part.get_content_disposition() and 'attachment' in part.get_content_disposition():
                    attachment_found = True  # 첨부파일이 있음을 설정
                    filename = part.get_filename()
                    attachment_data = part.get_payload(decode=True)
                    
                    if filename.lower().endswith(".pdf"):
                        attachment_text = extract_pdf_with_pdfplumber(attachment_data)
                    elif filename.lower().endswith(".docx"):
                        attachment_text = convert_docx_to_pdf_and_extract(attachment_data, filename)
        
        if not attachment_found:  # 첨부파일이 없는 경우
            print("첨부파일이 없습니다. 본문에서 링크를 검색합니다.")
            return extract_links_from_email(msg)  # 링크 추출
        
    except Exception as e:
        print(f"첨부파일 처리 중 오류 발생: {e}")
    return attachment_text

def extract_pdf_with_pdfplumber(pdf_data):
    """pdfplumber를 사용해 PDF에서 텍스트를 추출"""
    try:
        with pdfplumber.open(io.BytesIO(pdf_data)) as pdf:
            text = ''
            for page in pdf.pages:
                text += page.extract_text() + "\n"
        return text
    except Exception as e:
        print(f"PDF 처리 중 오류 발생: {e}")
        return ""

def convert_docx_to_pdf_and_extract(docx_data, filename):
    """DOCX 파일을 PDF로 변환한 후 PDF 처리"""
    try:
        temp_docx_filename = f"./{filename}"
        temp_pdf_filename = temp_docx_filename.replace(".docx", ".pdf")

        with open(temp_docx_filename, 'wb') as temp_docx_file:
            temp_docx_file.write(docx_data)

        word = win32.Dispatch('Word.Application')
        word.Visible = False
        doc = word.Documents.Open(os.path.abspath(temp_docx_filename))
        doc.SaveAs(os.path.abspath(temp_pdf_filename), FileFormat=17)
        doc.Close()

        with open(temp_pdf_filename, 'rb') as pdf_file:
            pdf_data = pdf_file.read()

        return extract_pdf_with_pdfplumber(pdf_data)

    except Exception as e:
        print(f"DOCX 파일을 PDF로 변환하는 중 오류 발생: {e}")
        return ""

    finally:
        if os.path.exists(temp_docx_filename):
            os.remove(temp_docx_filename)
        if os.path.exists(temp_pdf_filename):
            os.remove(temp_pdf_filename)

def extract_links_from_email(msg):
    """이메일 본문에서 HTML 링크(href)를 추출"""
    email_body = ""
    
    # HTML 또는 텍스트 형태의 본문을 추출
    if msg.is_multipart():
        for part in msg.iter_parts():
            if part.get_content_type() == 'text/html':
                html_content = part.get_payload(decode=True).decode(part.get_content_charset(), errors="replace")
                soup = BeautifulSoup(html_content, 'html.parser')
                email_body = soup.get_text()
                # href 속성으로부터 링크 추출
                links = [a['href'] for a in soup.find_all('a', href=True)]
                if links:
                    return f"본문에 포함된 링크:\n" + "\n".join(links)
            elif part.get_content_type() == 'text/plain':
                email_body = part.get_payload(decode=True).decode(part.get_content_charset(), errors="replace")
    
    # 만약 HTML이 없고 텍스트만 있다면, 정규 표현식을 사용하여 링크를 추출
    links = re.findall(r'(https?://\S+)', email_body)
    
    if links:
        return f"본문에 포함된 링크:\n" + "\n".join(links)
    else:
        return "본문에 링크가 없습니다."

# 첨부파일 본문 또는 링크 추출
attachment_body_or_links = extract_attachments(eml_file_path)
print(f"결과:\n{attachment_body_or_links}")


결과:



In [18]:
import openai
import re

# OpenAI API 키 설정
openai.api_key = '<REMOVED: set OPENAI_API_KEY env var>'  # 올바른 API 키 입력

def analyze_email_content(email_title, email_body, attachment_body=None):
    """이메일 제목, 본문, 첨부파일 본문 또는 링크 간의 연관성을 확인하고, 악성 여부를 판단"""
    
    # 첨부파일이 있는 경우
    if attachment_body:
        prompt = f"""
        Email Title: {email_title}
        Email Body: {email_body[:500]}  # 본문의 첫 500자만 사용
        Attachment Body: {attachment_body[:500]}  # 첨부파일의 첫 500자 사용
        
        Please analyze if the email title, the email body, and the attachment body are related. 
        Provide a brief summary of their relationship or similarity, indicating whether they are consistent with each other.
        Based on this analysis, do you think this email could be malicious or suspicious?
        """
    else:
        # 첨부파일이 없는 경우 이메일 본문에서 링크를 추출
        links = extract_links_from_email(email_body)
        if links:
            # 링크가 있을 경우
            prompt = f"""
            Email Title: {email_title}
            Email Body: {email_body[:500]}  # 본문의 첫 500자만 사용
            Links: {", ".join(links)}  # 본문에서 추출한 링크
            
            Please analyze if the email title, the email body, and the links provided are related. 
            Provide a brief summary of their relationship or similarity, and if the links seem suspicious or malicious.
            Based on this analysis, do you think this email could be malicious or suspicious?
            """
        else:
            # 링크와 첨부파일 둘 다 없을 경우
            prompt = f"""
            Email Title: {email_title}
            Email Body: {email_body[:500]}  # 본문의 첫 500자만 사용
            
            Please analyze if the email title and the email body are related. 
            Provide a brief summary of their relationship or similarity, indicating whether they are consistent with each other.
            Based on this analysis, do you think this email could be malicious or suspicious?
            """

    # OpenAI API 호출
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",  # 모델 사용
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},  # 시스템 역할 정의
            {"role": "user", "content": prompt}  # 사용자 입력
        ]
    )

    # 결과 반환
    result = response['choices'][0]['message']['content']
    is_malicious = assess_maliciousness(result)  # 악성 여부 판단
    return result, is_malicious

def extract_links_from_email(email_body):
    """이메일 본문에서 링크를 추출"""
    # 정규 표현식을 사용하여 링크를 추출
    links = re.findall(r'(https?://\S+)', email_body)
    return links

def assess_maliciousness(analysis_text):
    """악성 여부를 판단하는 간단한 함수"""
    # 악성일 가능성이 높은 단어 또는 구를 찾아서 판단
    malicious_keywords = ["suspicious", "malicious", "phishing", "dangerous"]
    
    for keyword in malicious_keywords:
        if keyword in analysis_text.lower():
            return "악성일 가능성이 있습니다."
    
    return "악성 이메일이 아닐 가능성이 높습니다."

# 이메일 제목, 본문, 첨부파일 본문 또는 링크 비교
email_title = "테스트 이메일 제목"
email_body = """
이메일 본문입니다. 다음은 링크입니다:
https://malicious-link.com
"""
attachment_body = None  # 첨부파일이 없다고 가정

# 분석 실행
comparison_result, is_malicious = analyze_email_content(email_title, email_body, attachment_body)
print("Comparison Result:")
print(comparison_result)
print("\n악성 여부 판단:")
print(is_malicious)


Comparison Result:
The email title "테스트 이메일 제목" and the email body "이메일 본문입니다" are both in Korean, suggesting they are related in terms of language. Additionally, the email body mentions a link, which is also included in the extracted links section as "https://malicious-link.com". This indicates that the email body content and the provided link are directly related. 

Given that the link provided is labeled as "malicious-link.com" and considering common phishing tactics where malicious actors attempt to trick recipients into clicking on harmful links, it is highly likely that the provided link is indeed suspicious or malicious. 

Based on this analysis, it is reasonable to conclude that this email could be malicious or suspicious, and caution should be exercised when interacting with its contents. It is advisable not to click on the provided link and to report such emails as potential phishing attempts.

악성 여부 판단:
악성일 가능성이 있습니다.
